> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
# ============================================================
# FIGURE 2 — PRIMARY CELLULAR TRAJECTORY ANALYSIS
# 782 ROI + common PC1 support
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    exist_ok=True
)

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 1. Load data
# ============================================================

# 大数据只读
adata = sc.read_h5ad(
    big_path,
    backed="r"
)

# 正式782 ROI对象
roi_ad = sc.read_h5ad(
    roi_path
)

obs = adata.obs

print("Large dataset:", adata.shape)
print("ROI dataset:", roi_ad.shape)


# ============================================================
# 2. Reconstruct the formal 782 ROI cells
# ============================================================

roi_cells = obs[
    obs["is_in_polygon"].to_numpy() == True
].copy()

roi_cells["roi_id"] = (
    roi_cells["polygon_flags"]
    .astype(str)
)

# 去掉重叠polygon
roi_cells = roi_cells[
    ~roi_cells["roi_id"].str.contains(
        ",",
        regex=False
    )
].copy()

# 只保留正式782 ROI里的ID
formal_roi_ids = set(
    roi_ad.obs_names.astype(str)
)

roi_cells = roi_cells[
    roi_cells["roi_id"].isin(
        formal_roi_ids
    )
].copy()

print("\nFormal ROI-associated cells:")
print(len(roi_cells))

print("Unique ROI:")
print(
    roi_cells["roi_id"].nunique()
)


# ============================================================
# 3. Define common PC1 support
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(
        ["min", "max"]
    )
)

pc_low = (
    pc_ranges["min"].max()
)

pc_high = (
    pc_ranges["max"].min()
)

print("\nPC1 ranges:")
print(pc_ranges)

print(
    "\nCommon PC1 support:",
    pc_low,
    "to",
    pc_high
)


# ============================================================
# 4. Define common-support ROIs
# ============================================================

roi_common_meta = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["ANCA", "SLE", "GBM"]
    )
    &
    roi_ad.obs[
        "PC1_crescent"
    ].between(
        pc_low,
        pc_high
    )
].copy()

common_roi_ids = (
    roi_common_meta
    .index
    .astype(str)
)

print("\nROI counts:")
print(
    roi_common_meta[
        "Disease"
    ].value_counts()
)

print("\nPatient counts:")
print(
    roi_common_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient_Sample_ID"
    ]
    .nunique()
)


# ============================================================
# 5. Restrict cell-level data to common-support ROIs
# ============================================================

roi_cells_common = roi_cells[
    roi_cells["roi_id"].isin(
        common_roi_ids
    )
].copy()

print(
    "\nCells in common-support ROIs:",
    len(roi_cells_common)
)


# ============================================================
# 6. Calculate ROI × cell-type counts
# ============================================================

cell_counts = (
    roi_cells_common
    .groupby(
        [
            "roi_id",
            "celltype_l1"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    "\nCell count matrix:",
    cell_counts.shape
)


# ============================================================
# 7. Convert counts to fractions
# ============================================================

cell_frac = (
    cell_counts.div(
        cell_counts.sum(
            axis=1
        ),
        axis=0
    )
)

# 按正式ROI顺序
usable_ids = [
    x for x in common_roi_ids
    if x in cell_frac.index
]

cell_frac = (
    cell_frac
    .loc[
        usable_ids
    ]
    .copy()
)


# ============================================================
# 8. Add metadata
# ============================================================

cell_frac["Disease"] = (
    roi_ad.obs.loc[
        usable_ids,
        "Disease"
    ]
    .astype(str)
    .values
)

cell_frac["Patient"] = (
    roi_ad.obs.loc[
        usable_ids,
        "Patient_Sample_ID"
    ]
    .astype(str)
    .values
)

cell_frac["PC1"] = (
    roi_ad.obs.loc[
        usable_ids,
        "PC1_crescent"
    ]
    .values
)


print(
    "\nAnalysis matrix:",
    cell_frac.shape
)


# ============================================================
# 9. Predefined cell types
# ============================================================

requested_types = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]

available_types = [
    ct for ct
    in requested_types
    if ct in cell_frac.columns
]

print(
    "\nCell types available:"
)

print(
    available_types
)

print(
    "n =",
    len(available_types)
)


# ============================================================
# 10. Prepare analysis data
# ============================================================

analysis_df = (
    cell_frac.copy()
)

analysis_df[
    "Disease"
] = pd.Categorical(
    analysis_df[
        "Disease"
    ],
    categories=[
        "ANCA",
        "SLE",
        "GBM"
    ]
)

# arcsine square-root transformation
for ct in available_types:

    analysis_df[
        ct + "_asin"
    ] = np.arcsin(
        np.sqrt(
            np.clip(
                analysis_df[ct],
                0,
                1
            )
        )
    )


# ============================================================
# 11. GLOBAL disease × PC1 spline tests
# ============================================================

global_rows = []

for ct in available_types:

    d = (
        analysis_df
        .copy()
    )

    # 每个患者总权重大致=1
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )

    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )

    formula = (
        f"{ct}_asin ~ "
        "bs(PC1, df=3, degree=3, "
        "include_intercept=False) "
        "* C(Disease)"
    )

    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
            d["Patient"]
        }
    )

    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )

    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    global_rows.append(
        [
            ct,
            float(
                wt.pvalue
            ),
            np.linalg.matrix_rank(
                fit.model.exog
            ),
            fit.model.exog.shape[1]
        ]
    )


global_results = pd.DataFrame(
    global_rows,
    columns=[
        "celltype",
        "trajectory_pvalue",
        "matrix_rank",
        "n_columns"
    ]
)


global_results[
    "FDR"
] = multipletests(
    global_results[
        "trajectory_pvalue"
    ],
    method="fdr_bh"
)[1]


global_results = (
    global_results
    .sort_values(
        "FDR"
    )
)


print(
    "\n================================"
)
print(
    "GLOBAL Disease × PC1 results"
)
print(
    "================================"
)

display(
    global_results
)


# ============================================================
# 12. Check model rank
# ============================================================

bad_global = global_results[
    global_results[
        "matrix_rank"
    ]
    !=
    global_results[
        "n_columns"
    ]
]

print(
    "\nNon-full-rank global models:"
)

display(
    bad_global
)


# ============================================================
# 13. Pairwise disease trajectory tests
# ============================================================

disease_pairs = [
    (
        "ANCA",
        "SLE"
    ),
    (
        "ANCA",
        "GBM"
    ),
    (
        "SLE",
        "GBM"
    )
]


pairwise_rows = []


for ct in available_types:

    for d1, d2 in disease_pairs:

        d = analysis_df[
            analysis_df[
                "Disease"
            ].isin(
                [
                    d1,
                    d2
                ]
            )
        ].copy()

        d[
            "Disease"
        ] = pd.Categorical(
            d[
                "Disease"
            ].astype(str),
            categories=[
                d1,
                d2
            ]
        )

        n_roi = (
            d.groupby(
                "Patient",
                observed=True
            )[
                "Patient"
            ]
            .transform(
                "size"
            )
        )

        d[
            "patient_weight"
        ] = (
            1.0 /
            n_roi
        )


        formula = (
            f"{ct}_asin ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


        fit = smf.wls(
            formula,
            data=d,
            weights=d[
                "patient_weight"
            ]
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups":
                d[
                    "Patient"
                ]
            }
        )


        interaction_terms = [
            term
            for term
            in fit.params.index
            if ":" in term
            and "C(Disease)" in term
            and "bs(PC1" in term
        ]


        R = np.zeros(
            (
                len(
                    interaction_terms
                ),
                len(
                    fit.params
                )
            )
        )


        for i, term in enumerate(
            interaction_terms
        ):

            R[
                i,
                fit.params.index
                .get_loc(term)
            ] = 1


        wt = fit.wald_test(
            R,
            scalar=True
        )


        pairwise_rows.append(
            [
                ct,
                f"{d1}_vs_{d2}",
                float(
                    wt.pvalue
                ),
                np.linalg.matrix_rank(
                    fit.model.exog
                ),
                fit.model.exog.shape[1]
            ]
        )


pairwise_results = pd.DataFrame(
    pairwise_rows,
    columns=[
        "celltype",
        "comparison",
        "pvalue",
        "matrix_rank",
        "n_columns"
    ]
)


# 每个comparison内部BH校正
pairwise_results[
    "FDR"
] = (
    pairwise_results
    .groupby(
        "comparison",
        observed=True
    )[
        "pvalue"
    ]
    .transform(
        lambda x:
        multipletests(
            x,
            method="fdr_bh"
        )[1]
    )
)


pairwise_results = (
    pairwise_results
    .sort_values(
        [
            "comparison",
            "FDR"
        ]
    )
)


print(
    "\n================================"
)
print(
    "PAIRWISE Disease × PC1 results"
)
print(
    "================================"
)

display(
    pairwise_results
)


# ============================================================
# 14. Pairwise rank check
# ============================================================

bad_pairwise = pairwise_results[
    pairwise_results[
        "matrix_rank"
    ]
    !=
    pairwise_results[
        "n_columns"
    ]
]


print(
    "\nNon-full-rank pairwise models:"
)

display(
    bad_pairwise
)


# ============================================================
# 15. Make a provisional 2 × 4 trajectory figure
# ============================================================

display_name = {
    "ANCA": "ANCA",
    "SLE": "LN",
    "GBM": "anti-GBM"
}

palette = {
    "ANCA": "#0072B2",
    "SLE": "#E69F00",
    "GBM": "#009E73"
}


fig, axes = plt.subplots(
    2,
    4,
    figsize=(
        16,
        8
    )
)

axes = (
    axes.flatten()
)


for ax, ct in zip(
    axes,
    available_types
):

    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        d = analysis_df[
            analysis_df[
                "Disease"
            ].astype(str)
            == disease
        ].sort_values(
            "PC1"
        )


        ax.scatter(
            d["PC1"],
            d[ct],
            s=10,
            alpha=0.12,
            color=palette[
                disease
            ]
        )


        if len(d) >= 10:

            sm = lowess(
                d[ct],
                d["PC1"],
                frac=0.55,
                return_sorted=True
            )

            ax.plot(
                sm[
                    :,
                    0
                ],
                sm[
                    :,
                    1
                ],
                linewidth=2,
                color=palette[
                    disease
                ],
                label=display_name[
                    disease
                ]
            )


    global_fdr = float(
        global_results.loc[
            global_results[
                "celltype"
            ]
            == ct,
            "FDR"
        ].iloc[0]
    )


    ax.set_title(
        f"{ct}   global FDR={global_fdr:.3g}"
    )

    ax.set_xlabel(
        "Crescent progression (PC1)"
    )

    ax.set_ylabel(
        "Cell fraction"
    )


axes[0].legend(
    frameon=False
)


plt.tight_layout(
    w_pad=2.2,
    h_pad=2.5
)


screen_path = (
    figdir /
    "Figure2_SCREENING_primary_PC1_cell_trajectories.png"
)


plt.savefig(
    screen_path,
    dpi=500,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 16. Save result tables
# ============================================================

global_results.to_csv(
    base /
    "figure2_primary_PC1_global_cell_trajectories.csv",
    index=False
)


pairwise_results.to_csv(
    base /
    "figure2_primary_PC1_pairwise_cell_trajectories.csv",
    index=False
)


analysis_df.to_csv(
    base /
    "figure2_primary_PC1_cell_fraction_data.csv"
)


print(
    "\n================================"
)

print(
    "FIGURE 2 ANALYSIS COMPLETE"
)

print(
    "================================"
)

print(
    "\nScreening figure:"
)

print(
    screen_path
)

print(
    "\nResults saved."
)

In [ ]:
# ============================================================
# FIGURE 2 — SHARED CELLULAR REMODELING
# Formal 782-ROI PC1 analysis
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests
from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(exist_ok=True)


# ============================================================
# 1. Read the data already saved by the previous analysis
# ============================================================

analysis_df = pd.read_csv(
    base / "figure2_primary_PC1_cell_fraction_data.csv",
    index_col=0
)


# ============================================================
# 2. Cell types used
# ============================================================

celltypes = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]


# ============================================================
# 3. Formal names / colors
# ============================================================

display_name = {
    "ANCA": "ANCA",
    "SLE": "LN",
    "GBM": "anti-GBM"
}

palette = {
    "ANCA": "#0072B2",
    "SLE": "#E69F00",
    "GBM": "#009E73"
}


# ============================================================
# 4. Make sure categorical disease is clean
# ============================================================

analysis_df["Disease"] = pd.Categorical(
    analysis_df["Disease"].astype(str),
    categories=[
        "ANCA",
        "SLE",
        "GBM"
    ]
)


# ============================================================
# 5. If asin columns are absent, recreate them
# ============================================================

for ct in celltypes:

    col = ct + "_asin"

    if col not in analysis_df.columns:

        analysis_df[col] = np.arcsin(
            np.sqrt(
                np.clip(
                    analysis_df[ct],
                    0,
                    1
                )
            )
        )


# ============================================================
# 6. Test SHARED PC1 progression effect
#
# Model:
# cell fraction ~ spline(PC1) + Disease
#
# Disease is adjusted for,
# but we no longer force a Disease × PC1 interaction.
# ============================================================

shared_rows = []


for ct in celltypes:

    d = analysis_df.copy()

    # each patient receives approximately equal total weight
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform("size")
    )

    d["patient_weight"] = (
        1.0 / n_roi
    )


    fit = smf.wls(
        (
            f"{ct}_asin ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "+ C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )


    # Jointly test the 3 spline terms
    pc1_terms = [
        term
        for term in fit.params.index
        if "bs(PC1" in term
        and "C(Disease)" not in term
    ]


    R = np.zeros(
        (
            len(pc1_terms),
            len(fit.params)
        )
    )


    for i, term in enumerate(pc1_terms):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1


    wt = fit.wald_test(
        R,
        scalar=True
    )


    shared_rows.append(
        [
            ct,
            float(wt.pvalue),
            np.linalg.matrix_rank(
                fit.model.exog
            ),
            fit.model.exog.shape[1]
        ]
    )


shared_results = pd.DataFrame(
    shared_rows,
    columns=[
        "celltype",
        "PC1_progression_pvalue",
        "matrix_rank",
        "n_columns"
    ]
)


shared_results["FDR"] = multipletests(
    shared_results["PC1_progression_pvalue"],
    method="fdr_bh"
)[1]


shared_results = shared_results.sort_values(
    "FDR"
)


print(
    "=================================="
)

print(
    "SHARED PC1 PROGRESSION EFFECTS"
)

print(
    "=================================="
)

display(
    shared_results
)


# ============================================================
# 7. Check model ranks
# ============================================================

print(
    "\nNon-full-rank models:"
)

display(
    shared_results[
        shared_results["matrix_rank"]
        !=
        shared_results["n_columns"]
    ]
)


# ============================================================
# 8. Plot shared cellular trajectories
# ============================================================

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8)
)

axes = axes.flatten()


for ax, ct in zip(
    axes,
    celltypes
):

    # --------------------------------
    # Disease-specific raw LOWESS
    # shown faintly
    # --------------------------------

    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        d = analysis_df[
            analysis_df["Disease"]
            .astype(str)
            == disease
        ].sort_values(
            "PC1"
        )


        ax.scatter(
            d["PC1"],
            d[ct],
            s=9,
            alpha=0.09,
            color=palette[disease]
        )


        sm = lowess(
            d[ct],
            d["PC1"],
            frac=0.55,
            return_sorted=True
        )


        ax.plot(
            sm[:, 0],
            sm[:, 1],
            linewidth=1.6,
            alpha=0.75,
            color=palette[disease],
            label=display_name[disease]
        )


    # --------------------------------
    # Overall shared LOWESS
    # --------------------------------

    pooled = analysis_df.sort_values(
        "PC1"
    )

    sm_all = lowess(
        pooled[ct],
        pooled["PC1"],
        frac=0.45,
        return_sorted=True
    )


    ax.plot(
        sm_all[:, 0],
        sm_all[:, 1],
        linewidth=3,
        color="black",
        label="Shared trend"
    )


    fdr = float(
        shared_results.loc[
            shared_results["celltype"]
            == ct,
            "FDR"
        ].iloc[0]
    )


    ax.set_title(
        f"{ct}   PC1 FDR={fdr:.3g}"
    )

    ax.set_xlabel(
        "Crescent progression (PC1)"
    )

    ax.set_ylabel(
        "Cell fraction"
    )


axes[0].legend(
    frameon=False,
    fontsize=8
)


plt.tight_layout(
    w_pad=2.2,
    h_pad=2.5
)


shared_fig = (
    figdir /
    "Figure2_SCREEN_shared_cellular_remodeling.png"
)


plt.savefig(
    shared_fig,
    dpi=500,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 9. Save statistics
# ============================================================

shared_results.to_csv(
    base /
    "figure2_shared_PC1_cellular_remodeling_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    shared_fig
)

In [ ]:
# ============================================================
# FIGURE 2 — FINAL VERSION
# Broadly conserved cellular remodeling along crescent PC1
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.nonparametric.smoothers_lowess import lowess


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    exist_ok=True
)


# ============================================================
# 1. Read saved Figure 2 data/results
# ============================================================

analysis_df = pd.read_csv(
    base /
    "figure2_primary_PC1_cell_fraction_data.csv",
    index_col=0
)

shared_results = pd.read_csv(
    base /
    "figure2_shared_PC1_cellular_remodeling_results.csv"
)

interaction_results = pd.read_csv(
    base /
    "figure2_primary_PC1_global_cell_trajectories.csv"
)


print("Analysis data:")
print(analysis_df.shape)

print("\nShared progression results:")
display(shared_results)

print("\nDisease × progression results:")
display(interaction_results)


# ============================================================
# 2. Disease names / colors
# ============================================================

display_name = {
    "ANCA": "ANCA",
    "SLE": "LN",
    "GBM": "anti-GBM"
}

palette = {
    "ANCA": "#0072B2",
    "SLE": "#E69F00",
    "GBM": "#009E73"
}


# ============================================================
# 3. Cell types
# ============================================================

all_celltypes = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]

# Main representative panels
trajectory_types = [
    "MAC",
    "Mono",
    "FIB",
    "EC",
    "POD"
]


# ============================================================
# 4. Prepare significance table
# ============================================================

shared_map = (
    shared_results
    .set_index("celltype")["FDR"]
)

interaction_map = (
    interaction_results
    .set_index("celltype")["FDR"]
)


sig_matrix = []

for ct in all_celltypes:

    shared_fdr = float(
        shared_map.loc[ct]
    )

    interaction_fdr = float(
        interaction_map.loc[ct]
    )

    sig_matrix.append(
        [
            -np.log10(
                max(
                    shared_fdr,
                    1e-300
                )
            ),
            -np.log10(
                max(
                    interaction_fdr,
                    1e-300
                )
            )
        ]
    )


sig_matrix = np.array(
    sig_matrix
)

# 为了颜色不会被极小P值完全撑爆，
# 图形显示上最多显示 -log10(FDR)=20
sig_matrix_plot = np.clip(
    sig_matrix,
    0,
    20
)


# ============================================================
# 5. Create complete Figure 2
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 9)
)

axA = axes[0, 0]
axB = axes[0, 1]
axC = axes[0, 2]

axD = axes[1, 0]
axE = axes[1, 1]
axF = axes[1, 2]


# ============================================================
# PANEL A
# Shared progression vs Disease × PC1
# ============================================================

ax = axA

im = ax.imshow(
    sig_matrix_plot,
    aspect="auto",
    cmap="viridis"
)

ax.set_yticks(
    np.arange(
        len(all_celltypes)
    )
)

ax.set_yticklabels(
    all_celltypes
)

ax.set_xticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "Shared\nPC1 effect",
        "Disease × PC1\ninteraction"
    ]
)

ax.set_title(
    "A  Conserved vs disease-specific remodeling",
    loc="left",
    fontsize=12
)


# FDR=0.05 threshold
threshold = -np.log10(
    0.05
)


# Add actual FDR text
for i, ct in enumerate(
    all_celltypes
):

    q_shared = float(
        shared_map.loc[ct]
    )

    q_int = float(
        interaction_map.loc[ct]
    )

    texts = [
        q_shared,
        q_int
    ]

    for j, q in enumerate(
        texts
    ):

        if q < 0.001:

            label = (
                f"{q:.1e}"
            )

        else:

            label = (
                f"{q:.3f}"
            )

        # 根据背景自动选择文字颜色
        value = (
            sig_matrix_plot[
                i,
                j
            ]
        )

        text_color = (
            "white"
            if value > 8
            else "black"
        )

        ax.text(
            j,
            i,
            label,
            ha="center",
            va="center",
            fontsize=7.5,
            color=text_color
        )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "-log10(FDR)"
)


# ============================================================
# Helper function for trajectory panels
# ============================================================

def plot_trajectory(
    ax,
    celltype,
    panel_letter
):

    # --------------------------------
    # Disease-specific curves
    # --------------------------------

    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        d = analysis_df[
            analysis_df[
                "Disease"
            ].astype(str)
            == disease
        ].sort_values(
            "PC1"
        )


        ax.scatter(
            d["PC1"],
            d[celltype],
            s=10,
            alpha=0.09,
            color=palette[
                disease
            ]
        )


        sm = lowess(
            d[celltype],
            d["PC1"],
            frac=0.55,
            return_sorted=True
        )


        ax.plot(
            sm[:, 0],
            sm[:, 1],
            linewidth=1.6,
            alpha=0.75,
            color=palette[
                disease
            ],
            label=display_name[
                disease
            ]
        )


    # --------------------------------
    # Overall pooled/shared trend
    # --------------------------------

    pooled = (
        analysis_df
        .sort_values(
            "PC1"
        )
    )

    sm_all = lowess(
        pooled[celltype],
        pooled["PC1"],
        frac=0.45,
        return_sorted=True
    )


    ax.plot(
        sm_all[:, 0],
        sm_all[:, 1],
        linewidth=3,
        color="black",
        label="Shared trend"
    )


    # --------------------------------
    # FDR values
    # --------------------------------

    shared_fdr = float(
        shared_map.loc[
            celltype
        ]
    )

    interaction_fdr = float(
        interaction_map.loc[
            celltype
        ]
    )


    ax.set_title(
        (
            f"{panel_letter}  {celltype}\n"
            f"PC1 FDR={shared_fdr:.2g}; "
            f"interaction FDR={interaction_fdr:.2g}"
        ),
        loc="left",
        fontsize=11
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )

    ax.set_ylabel(
        "Cell fraction"
    )


# ============================================================
# PANEL B — MAC
# ============================================================

plot_trajectory(
    axB,
    "MAC",
    "B"
)

axB.legend(
    frameon=False,
    fontsize=8
)


# ============================================================
# PANEL C — Mono
# ============================================================

plot_trajectory(
    axC,
    "Mono",
    "C"
)


# ============================================================
# PANEL D — FIB
# ============================================================

plot_trajectory(
    axD,
    "FIB",
    "D"
)


# ============================================================
# PANEL E — EC
# ============================================================

plot_trajectory(
    axE,
    "EC",
    "E"
)


# ============================================================
# PANEL F — POD
# ============================================================

plot_trajectory(
    axF,
    "POD",
    "F"
)


# ============================================================
# 6. Final layout
# ============================================================

plt.tight_layout(
    w_pad=2.5,
    h_pad=3.0
)


# ============================================================
# 7. Save PNG
# ============================================================

png_path = (
    figdir /
    "Figure2_COMPLETE_shared_cellular_remodeling.png"
)

plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)


# ============================================================
# 8. Save PDF
# ============================================================

pdf_path = (
    figdir /
    "Figure2_COMPLETE_shared_cellular_remodeling.pdf"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 9. Confirm
# ============================================================

print(
    "\nFigure 2 saved successfully!"
)

print(
    "\nPNG:"
)

print(
    png_path
)

print(
    "\nPDF:"
)

print(
    pdf_path
)